In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

%matplotlib inline
plt.rcParams.update({
    'font.size':         8,
    'figure.dpi':        300,
    'svg.fonttype':      'none',
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Roboto'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src.notebook import load_notebook_data
from src.timescales import (create_ints_dataframe_custom_grouping, convert_data_dict,
                             average_data_dict_converted, compute_deflections)
from src.plotting import register_custom_colormaps, show_colormaps, save_panel
from src.nulls import null_basis, generate_surrogates, require_full_parcellation
from snaplab_tools.gams import detect_change_point, partial_r2_smooth
from src.gams import (fit_all_panels, apply_region_mask, get_series_colors, get_draw_order,
                      style_ax, compute_deriv_sa_corr, bootstrap_gam_maps,
                      add_sa_decile_colorbar, plot_r_age, deriv_map_spatial_null_p,
                      SIGNAL_TYPES, signal_labels, gam_variant_tags, gam_cache_path,
                      build_nuisance)

# Load data

In [ ]:
dataset       = 'HCP-D' # 'HCP-D' 'PNC'
which_modality = 'FMRI'
n_regions = 400

if dataset == 'HCP-D':
    # output_type   = 'timescale_auc_zero_crossing'
    output_type   = 'timescale_auc_zero_crossing_bp-0.01-0.08hz'
elif dataset == 'PNC':
    output_type   = 'timescale_auc_zero_crossing'
    # output_type   = 'timescale_zero_crossing'
    # output_type   = 'timescale_zero_crossing_robust_nc2'
    # output_type   = 'timescale_zero_crossing_robust_nc3'

notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   output_type=output_type, use_resids=True,
                                   load_subject_data=False, n_regions=n_regions)

In [ ]:
timescales_processor = notebook_data['timescales_processor']
data_dict            = notebook_data['data_dict']
results_dir          = notebook_data['results_dir']
df_neuro             = notebook_data['supporting_data']['df_neuro']

sa_axis = df_neuro['SA_Axis'].values

nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/{dataset}'
os.makedirs(nas_scratch_dir, exist_ok=True)

In [ ]:
sa_axis.shape

# Prepare data

In [ ]:
timescales_processor.match_registry_to_behavior()
df_behavior = timescales_processor.df_behavior_matched.copy()

if dataset == 'HCP-D':
    rest_scan = 'task-Rest'
    # task_scan = 'tfMRICARITPA'
    # task_scan = 'tfMRIEMOTIONPA'
    task_scan = 'tfMRIGUESSINGPA'
elif dataset == 'PNC':
    rest_scan = 'rest'
    task_scan = 'frac2back'

data_dict_tsnr      = notebook_data['data_dict_tsnr']
data_dict_converted = convert_data_dict(data_dict, timescales_processor.scan_registry)
data_dict_centiles  = convert_data_dict(data_dict, timescales_processor.scan_registry,
                                        brain_map=sa_axis)
data_dict_converted = average_data_dict_converted(data_dict_converted, average_all_rest=True)
data_dict_centiles  = average_data_dict_converted(data_dict_centiles,  average_all_rest=True)
df_tsnr             = create_ints_dataframe_custom_grouping(data_dict_tsnr, average_all_rest=True,
                                                            n_regions=n_regions)

# Fit GAMs

In [ ]:
df_behavior

In [ ]:
# ── GAM + bootstrap configuration ────────────────────────────────
# Fitting recipe (switch fit_recipe). Both are kept available:
#   'historical' -> n_splines=4, lam=0.1 FIXED (the original recipe; writes the original cache file)
#   'gcv'        -> n_splines=5, lam selected per region by GCV (flexible; separate cache file)
# GCV self-regulates by SNR: the high-SNR decile curves gain real curvature (2nd-derivative
# structure) while individual regions self-limit to ~2-3 effective DOF, so the regional analysis
# stays honest. lam is selected once on the full sample and frozen for the bootstrap.
fit_recipe = 'gcv'

# Deflection definition: absolute=True -> |task - rest| (magnitude of task-evoked modulation,
# direction discarded); False -> signed (task - rest). Only the 'deflection' signal depends on
# this -- the rest and task fits are bit-identical either way -- but the stage boundary is
# detected on the deflection fit, so it still reaches the rest/task panels through the boundary
# annotation and (in fig5) the stage partition. Recorded in the cache filename and save_dict.
absolute            = True
# Demeaning: if True, every series (region / decile) is centred on its own mean before its GAM is
# fit, so the predictions are deviations from that series' grand mean and all curves share a common
# zero. If False, the fits keep their native offsets (predictions carry each series' absolute
# level). Derivatives are unchanged either way; predictions, partial R^2, and anything detected on
# the predictions (the change-point boundary) are not. This one flag drives the fits, the bootstrap,
# the partial R^2, and the figure styling, and is recorded in both the cache filename and save_dict.
demean              = True

# fit_recipe + absolute + demean ARE the run's identity. src.gams owns the mapping from those
# three to n_splines / lam / every filename tag, so fig5 resolves the identical path from the
# identical inputs instead of the two notebooks hand-building the same strings.
_tags        = gam_variant_tags(fit_recipe, absolute, demean)
n_splines    = _tags['n_splines']
lam          = _tags['lam']
recipe_tag   = _tags['recipe_tag']
absolute_tag = _tags['absolute_tag']
demean_tag   = _tags['demean_tag']
_absolute_cfg, _demean_cfg = absolute, demean   # snapshots for the cache-vs-config check below

# ── How panel E judges significance ──────────────────────────────
# The two tests answer DIFFERENT questions, so this is a choice about what panel E claims:
#   'bootstrap' -- resample SUBJECTS, refit every region, ask "is the coupling reliably
#                  non-zero given sampling error?". Blind to spatial autocorrelation.
#   'spatial'   -- hold the fit fixed, surrogate the S-A AXIS with matched spatial
#                  autocorrelation (BrainSMASH), ask "is this coupling stronger than the axis's
#                  own smoothness would produce?" (fig5's test). Blind to sampling error.
#   'both'      -- bootstrap band, spatial-null significance marks. The only setting that does
#                  not overclaim, and the default.
# Both are always computed and exported, so switching is a re-plot, not a re-fit.
SIG_METHOD     = 'bootstrap'        # 'bootstrap' | 'spatial' | 'both'
SA_CORR_METHOD = 'pearson'     # correlation for the derivative x S-A coupling ('pearson'/'spearman')
N_PERMS        = 10000         # BrainSMASH surrogates of the S-A axis. Match fig5's N_PERMS and
                               # the cached 'sa_axis' nulls are shared rather than regenerated.
SIG_ALPHA      = 0.05          # threshold on the spatial-null p
if SIG_METHOD not in ('bootstrap', 'spatial', 'both'):
    raise ValueError(f"SIG_METHOD must be 'bootstrap', 'spatial' or 'both'; got {SIG_METHOD!r}")
n_boot              = 500      # bootstrap iterations (derivative × S-A CI + fig5 stage maps)
boot_seed           = 42        # resampling seed (shared by all bootstrap reductions)
N_JOBS              = -1        # joblib workers for the bootstrap (-1 = all cores)

# Data-driven stage boundary: 'deriv1' (rate reorganization) is stable across fitting recipes;
# deriv2 (acceleration) is near-degenerate at the regional level (its boundary drifts with
# flexibility), so it is only REPORTED as a robustness CI, not used to define the stage maps.
stage_boundary_signal   = 'deriv1'       # drives the exported stage maps ('deriv1'|'deriv2'|'pred')
boundary_report_signals = ('deriv2', 'pred')    # extra signals to report a boundary CI for (maps unaffected)

# Number of change points to detect for the boundary annotation (the figure draws this many
# vertical age lines). n_bkps=1 -> the single stage boundary (two stages). n_bkps>1 uses ruptures
# (exploratory; the smooth GAM signals tend to over-segment) and does NOT change the two-stage
# stage maps exported for fig5 — those still use the single-boundary detection.
n_bkps = 1

# The GAM fits + bootstrap are slow, so their outputs are cached to gam_cache_file and reloaded on
# re-runs. Set FORCE_RERUN = True to recompute from scratch and overwrite the cache.
# NOTE: regenerate the cache (True) after changing the fitting recipe, the absolute or demean
# flags, or the stage-boundary definition — the exported stage maps and fig5's inputs depend on
# them. recipe_tag, absolute_tag and demean_tag keep each combination in its own file.
FORCE_RERUN    = False
gam_out_dir    = os.path.join(timescales_processor.output_dir, 'results', dataset, 'gam')
os.makedirs(gam_out_dir, exist_ok=True)
gam_cache_file = gam_cache_path(gam_out_dir, dataset, n_regions, task_scan, _tags)

# Shared spatial-null basis (same call fig5 makes, so both notebooks hit the same cache).
distance_matrix, hemi_labels, cache_dir = null_basis(timescales_processor, dataset, n_regions)
print(f'GAM cache -> {os.path.basename(gam_cache_file)}')
print(f'Panel E significance: {SIG_METHOD} ({SA_CORR_METHOD}); surrogate cache {cache_dir}')

In [ ]:
# Load the cached GAM + bootstrap outputs if present, else compute them from scratch and save.
# SIGNAL_TYPES (rest/task/deflection) comes from src.gams so fig5 uses the same vocabulary. Note
# it is NOT snaplab_tools.gams.SIGNALS, which names the curve a change point is detected on.
SIGNALS = SIGNAL_TYPES

if (not FORCE_RERUN) and os.path.exists(gam_cache_file):
    save_dict = np.load(gam_cache_file, allow_pickle=True).item()
    print(f'Loaded cached GAM + bootstrap outputs <- {gam_cache_file}')
    print('  (set FORCE_RERUN = True in the config cell above to recompute and overwrite)')
else:
    # ===== Region- and decile-level data =====
    rest_full = data_dict_converted[rest_scan]
    try:
        task_full = data_dict_converted[task_scan]
    except (KeyError, TypeError):
        task_full = data_dict_converted[rest_scan]
    n_subs = task_full.shape[0]
    n_regions = task_full.shape[1]

    # Covariate data (age smooth + nuisance: sex, head motion, each subject's mean rest INT).
    # build_nuisance is shared with fig5, whose bootstrap has to reproduce this exact fit.
    age = df_behavior['age'].values
    nuisance = build_nuisance(df_behavior, rest_full, dataset)
    age_grid = np.linspace(age.min(), age.max(), 200)

    deflection_full = compute_deflections(rest_full, task_full, absolute=absolute)

    rest_dec = data_dict_centiles[rest_scan]
    try:
        task_dec = data_dict_centiles[task_scan]
    except (KeyError, TypeError):
        task_dec = data_dict_centiles[rest_scan]
    deflection_dec = compute_deflections(rest_dec, task_dec, absolute=absolute)
    n_deciles = rest_dec.shape[1]

    # ===== Fit GAMs (decile + region). lam is a float (fixed) or 'gcv' (per-series grid search).
    # demean comes from the config cell and is used identically here, in the bootstrap, and in the
    # partial R^2 below (partial_r2_smooth requires the same flag the fits used). =====
    results_dec = fit_all_panels(age, rest_dec, task_dec, deflection_dec, age_grid,
                                 n_splines=n_splines, lam=lam, demean=demean, show_ci=False,
                                 nuisance=nuisance)

    apply_tsnr_mask = False  # set True to exclude low-tSNR regions
    if apply_tsnr_mask:
        region_mask = (df_tsnr[task_scan] < np.quantile(df_tsnr[task_scan], q=0.1)).values
    else:
        region_mask = None
    rest_reg, task_reg, defl_reg, sa_axis_reg = apply_region_mask(
        region_mask, rest_full, task_full, deflection_full, sa_axis=sa_axis)
    results_reg = fit_all_panels(age, rest_reg, task_reg, defl_reg, age_grid,
                                 n_splines=n_splines, lam=lam, demean=demean, show_ci=False,
                                 nuisance=nuisance)
    reg_data = {'rest': rest_reg, 'task': task_reg, 'deflection': defl_reg}
    print(f'GAMs fit ({fit_recipe}, absolute={absolute}, demean={demean}). '
          f'Decile {deflection_dec.shape} | '
          f'Regional {defl_reg.shape} | Nuisance {None if nuisance is None else nuisance.shape}')
    if lam == 'gcv':
        _ed = np.array([g.statistics_['edof'] for g in results_reg['deflection']['gams'] if g is not None])
        _lam_med = np.nanmedian(results_reg['deflection']['lams'])
        print(f'  GCV lam median {_lam_med:.2f} | regional edof median {np.median(_ed):.2f} '
              f'(incl. nuisance terms)')

    # ===== Bootstrap: each subject-resample refit is reduced ONCE to the derivative × S-A axis
    # correlation curve (panel E's CI), the stage-1/stage-2 net-change maps (exported for Figure 5's
    # Yeo-7 enrichment, so fig5 does zero GAM fitting), AND a data-driven stage boundary detected
    # per resample from stage_boundary_signal (maps integrate over each resample's own window).
    # boundary_report_signals are detected too, for robustness CIs. Parallel over resamples.
    #
    # ALL THREE signals are bootstrapped, because panel E can plot any of them and its CI band --
    # plus the significance shading and crossing-age annotations derived from it -- must belong to
    # the curve actually drawn. Only the deflection run also produces the stage maps and boundaries:
    # those define fig5's inputs and are deliberately deflection-specific. All three share
    # boot_seed, so they are bootstrapped over identical subject resamples. =====
    _boundary_signals = (stage_boundary_signal,) + tuple(boundary_report_signals)
    boot_curves, r_ci_lo, r_ci_hi = {}, {}, {}
    for sig in SIGNALS:
        _is_defl = sig == 'deflection'
        # Freeze this signal's per-region smoothing from its own full-sample fit (the GCV-selected
        # lam, or the fixed scalar) so every resample reuses it rather than re-selecting it.
        _lam = results_reg[sig]['lams']
        _boot = bootstrap_gam_maps(
            reg_data[sig], age, age_grid, sa_axis_vals=sa_axis_reg,
            inflection_age=stage_boundary_signal if _is_defl else None,
            report_boundary_signals=boundary_report_signals if _is_defl else (),
            n_splines=n_splines, lam=_lam, demean=demean, nuisance=nuisance, annualized=True,
            n_boot=n_boot, method='pearson', random_state=boot_seed, n_jobs=N_JOBS)
        boot_curves[sig] = _boot['sa_curves']
        r_ci_lo[sig] = np.nanpercentile(boot_curves[sig], 2.5,  axis=0)
        r_ci_hi[sig] = np.nanpercentile(boot_curves[sig], 97.5, axis=0)
        if _is_defl:
            frozen_lam = _lam                                        # exported for fig5
            boot_stage1, boot_stage2 = _boot['stage1'], _boot['stage2']
            boot_boundary = {s: _boot[f'boundary_{s}'] for s in _boundary_signals}

    # Point-estimate boundary on the full-sample regional DEFLECTION fit: defines the stage boundary
    # fig5 uses for its observed maps and annotates every panel of this figure (including when
    # signal_type is rest/task -- the boundary is a fixed deflection-derived reference, not a
    # per-signal quantity); the bootstrap distribution above gives its CI.
    boundary_pt    = {s: detect_change_point(results_reg['deflection']['pred'], age_grid, signal=s)
                      for s in _boundary_signals}
    inflection_age = float(boundary_pt[stage_boundary_signal]['location'])
    # Multi-change-point boundaries for the figure annotation (draws n_bkps vertical lines); n_bkps=1
    # reproduces the single stage boundary. Detected on the same regional signal that drives the maps.
    boundary_ages  = detect_change_point(results_reg['deflection']['pred'], age_grid,
                                           signal=stage_boundary_signal, n_bkps=n_bkps)['locations']
    # Each signal's OWN boundary (point estimate only, same detector). fig5 can partition rest/task
    # by these instead of the shared deflection boundary; 'deflection' here equals inflection_age.
    signal_inflection = {sig: float(detect_change_point(results_reg[sig]['pred'], age_grid,
                                                        signal=stage_boundary_signal)['location'])
                         for sig in SIGNALS}

    # ===== Spatial null for panel E: surrogate the S-A AXIS, hold the fit fixed. Complements
    # the subject bootstrap above rather than replacing it (different question -- see the config
    # cell). Cheap once the surrogates exist: one matmul per signal. =====
    require_full_parcellation(
        sa_axis_reg.shape[0], n_regions, 'the regional fit',
        'The S-A axis surrogates span the whole parcellation, so set apply_tsnr_mask = False.')
    sa_axis_surr = generate_surrogates(
        sa_axis_reg, name='sa_axis', n_regions=n_regions, n_perms=N_PERMS,
        distance_matrix=distance_matrix, hemi=hemi_labels, cache_dir=cache_dir, seed=0)
    sa_null_p = {}
    for sig in SIGNALS:
        _r_chk, sa_null_p[sig] = deriv_map_spatial_null_p(
            results_reg[sig]['deriv'], sa_axis_reg, sa_axis_surr, method=SA_CORR_METHOD)
        # deriv_map_spatial_null_p restricts to parcels finite across the map, the whole
        # derivative curve AND every surrogate, so its r can in principle differ from the
        # compute_deriv_sa_corr curve the figure draws. Report the gap rather than assume none.
        _gap = np.nanmax(np.abs(_r_chk - compute_deriv_sa_corr(
            results_reg[sig]['deriv'], sa_axis_reg, method=SA_CORR_METHOD)))
        print(f'  spatial null [{sig}]: {int((sa_null_p[sig] < SIG_ALPHA).sum())}/'
              f'{len(age_grid)} age points p<{SIG_ALPHA} (max |r| gap vs plotted curve {_gap:.2e})')

    print(f'Bootstrap done ({n_boot} iters × {len(SIGNALS)} signals, n_jobs={N_JOBS}). '
          '95% CI at age midpoint:')
    for sig in SIGNALS:
        print(f'  {sig:<11}[{r_ci_lo[sig][100]:.3f}, {r_ci_hi[sig][100]:.3f}]  '
              f'own boundary {signal_inflection[sig]:.2f} yr')
    _b = boot_boundary[stage_boundary_signal]
    print(f'Data-driven stage boundary [{stage_boundary_signal}]: {inflection_age:.2f} yr '
          f'(point est.), 95% CI [{np.nanpercentile(_b, 2.5):.2f}, {np.nanpercentile(_b, 97.5):.2f}] yr.')
    if n_bkps > 1:
        print(f'  n_bkps={n_bkps} boundaries (annotation only): {[round(a, 2) for a in boundary_ages]} yr')
    for s in boundary_report_signals:
        _bs = boot_boundary[s]
        print(f'  boundary [{s}] (robustness): {boundary_pt[s]["location"]:.2f} yr (point est.), '
              f'95% CI [{np.nanpercentile(_bs, 2.5):.2f}, {np.nanpercentile(_bs, 97.5):.2f}] yr.')
    print(f'Bootstrap stage maps for fig5: stage1 {boot_stage1.shape}, stage2 {boot_stage2.shape}.')

    # ===== Assemble outputs (figure inputs + cross-notebook export for fig5) =====
    save_dict = {
        'age_grid':    age_grid,
        'sa_axis_reg': sa_axis_reg,
        'task_scan':   task_scan,
        'n_deciles':   n_deciles,
    }
    for sig in SIGNALS:
        save_dict[f'{sig}_deriv']     = results_reg[sig]['deriv']
        save_dict[f'{sig}_pred']      = results_reg[sig]['pred']
        save_dict[f'{sig}_deriv_dec'] = results_dec[sig]['deriv']
        save_dict[f'{sig}_pred_dec']  = results_dec[sig]['pred']
        save_dict[f'{sig}_partial_r2'] = partial_r2_smooth(
            reg_data[sig], age, results_reg[sig]['gams'], nuisance=nuisance, demean=demean
        )
        # Panel E inputs, per signal: observed r curve + its own bootstrap band.
        save_dict[f'{sig}_r_vals']      = compute_deriv_sa_corr(
            results_reg[sig]['deriv'], sa_axis_reg, method='pearson')
        save_dict[f'{sig}_boot_curves'] = boot_curves[sig]
        save_dict[f'{sig}_r_ci_lo']     = r_ci_lo[sig]
        save_dict[f'{sig}_r_ci_hi']     = r_ci_hi[sig]
        save_dict[f'{sig}_sa_null_p']   = sa_null_p[sig]   # pointwise BrainSMASH p per age
        # fig5 inputs, per signal: the subject × region matrix it re-fits per bootstrap resample,
        # the frozen smoothing for those refits, and this signal's own stage boundary.
        save_dict[f'{sig}_reg']            = reg_data[sig]
        save_dict[f'{sig}_lams']           = results_reg[sig]['lams']
        save_dict[f'{sig}_inflection_age'] = signal_inflection[sig]
    # Un-prefixed aliases: these keys have always meant deflection; kept for downstream readers.
    save_dict['r_vals']         = save_dict['deflection_r_vals']
    save_dict['boot_curves']    = save_dict['deflection_boot_curves']
    save_dict['r_ci_lo']        = r_ci_lo['deflection']
    save_dict['r_ci_hi']        = r_ci_hi['deflection']
    save_dict['boot_stage1']    = boot_stage1     # (n_boot, n_regions) fig5 enrichment bootstrap
    save_dict['boot_stage2']    = boot_stage2
    save_dict['inflection_age'] = inflection_age  # deflection boundary (yr); fig5's default partition
    save_dict['stage_boundary_signal'] = stage_boundary_signal
    save_dict['boundary_ages']  = boundary_ages   # n_bkps annotation boundaries (yr)
    save_dict['n_bkps']         = n_bkps
    for s in _boundary_signals:
        save_dict[f'boundary_{s}']    = boot_boundary[s]                       # (n_boot,) boundary CI
        save_dict[f'boundary_pt_{s}'] = float(boundary_pt[s]['location'])  # full-sample point est.
    save_dict['n_boot']         = n_boot
    save_dict['n_perms']        = N_PERMS                                      # spatial-null count
    save_dict['sa_corr_method'] = SA_CORR_METHOD
    save_dict['boot_seed']      = boot_seed
    save_dict['fit_recipe']     = fit_recipe                                   # provenance
    save_dict['n_splines']      = n_splines
    save_dict['demean']         = demean                                       # centring used by every fit here
    save_dict['absolute']       = absolute                                     # |task-rest| vs signed (task-rest)
    # fig5 works at the full parcellation (surface plots + BrainSMASH surrogates), so it refuses
    # a masked export; record the state rather than letting the length mismatch surface as a
    # broadcasting error several cells downstream.
    save_dict['region_mask_applied'] = region_mask is not None
    save_dict['gam_lam']        = lam if isinstance(lam, str) else float(lam)
    save_dict['frozen_lam']     = frozen_lam                                   # (n_regions,) deflection smoothing used
    save_dict['rest_reg']    = rest_reg
    save_dict['defl_reg']    = defl_reg
    save_dict['sa_axis']     = sa_axis
    save_dict['df_behavior'] = df_behavior

    np.save(gam_cache_file, save_dict)
    print(f'Computed and saved GAM + bootstrap outputs -> {gam_cache_file}')
    for sig in SIGNALS:
        r2 = save_dict[f'{sig}_partial_r2']
        print(f"  {sig}_partial_r2: shape={r2.shape}, mean={np.nanmean(r2):.4f}, max={np.nanmax(r2):.4f}")

In [ ]:
# Reconstruct the figure inputs from save_dict (identical whether loaded or freshly computed).
age_grid    = save_dict['age_grid']
sa_axis_reg = save_dict['sa_axis_reg']
n_deciles   = save_dict['n_deciles']

# Panel E inputs, keyed by signal so the verdict always belongs to the curve being plotted.
# Older caches only bootstrapped deflections: rest/task come back as None there and the figure
# falls back to a bare curve rather than borrowing the deflection CI.
r_ci_lo   = {sig: save_dict.get(f'{sig}_r_ci_lo') for sig in SIGNALS}
r_ci_hi   = {sig: save_dict.get(f'{sig}_r_ci_hi') for sig in SIGNALS}
sa_null_p = {sig: save_dict.get(f'{sig}_sa_null_p') for sig in SIGNALS}
if r_ci_lo['deflection'] is None:                      # pre-per-signal cache
    r_ci_lo['deflection'] = save_dict['r_ci_lo']
    r_ci_hi['deflection'] = save_dict['r_ci_hi']

# The options the cached fits actually used. Read back from save_dict (not the config cell) so a
# loaded cache is always plotted consistently with how it was built; caches predating these keys
# were demeaned and absolute. The tags are re-derived from them so figure filenames match the
# fits they show, and a config cell edited without a refit is reported rather than silently used.
demean      = bool(save_dict.get('demean', True))
absolute    = bool(save_dict.get('absolute', True))
_tags_cache = gam_variant_tags(save_dict.get('fit_recipe', fit_recipe), absolute, demean)
variant_tag = _tags_cache['variant_tag']
for _name, _cached, _cfg in [('demean', demean, _demean_cfg), ('absolute', absolute, _absolute_cfg)]:
    if _cached != _cfg:
        print(f'WARNING: cache was built with {_name}={_cached} but the config cell says '
              f'{_name}={_cfg}. Set FORCE_RERUN = True to rebuild.')

# Data-driven stage boundary (point estimate + subject-bootstrap 95% CI) for annotation. Always
# detected on the DEFLECTION fit, so it is a fixed reference on every panel, not a per-signal value.
boundary_signal = save_dict.get('stage_boundary_signal', 'deriv1')
boundary_pt_age = save_dict.get('inflection_age')                  # float, data-driven point estimate (yr)
_bd             = save_dict.get(f'boundary_{boundary_signal}')     # (n_boot,) or None
boundary_ci     = np.nanpercentile(_bd, [2.5, 97.5]) if _bd is not None else None
# n_bkps change-point boundaries drawn as vertical lines (falls back to the single boundary)
boundary_ages   = save_dict.get('boundary_ages',
                                [boundary_pt_age] if boundary_pt_age is not None else [])

results_reg = {sig: {'deriv': save_dict[f'{sig}_deriv'],     'pred': save_dict[f'{sig}_pred']}
               for sig in SIGNALS}
results_dec = {sig: {'deriv': save_dict[f'{sig}_deriv_dec'], 'pred': save_dict[f'{sig}_pred_dec']}
               for sig in SIGNALS}

cmap          = register_custom_colormaps()['snap_sydnor_sa']   # registers all custom maps; sydnor_sa colours the decile lines + S-A colorbar
series_colors = get_series_colors('decile', n_deciles, cmap)
draw_order    = get_draw_order('decile', n_deciles)

# Figure 4

In [ ]:
# --- Parameters ---
signal_type = 'deflection'  # 'rest', 'task', or 'deflection'
plot_level  = 'region'      # 'region' or 'decile'
heatmap_cmap = 'snap_blue_pink'  # diverging map for panel A (blue<->red; avoids the orange/purple of the S-A axis). preview all via show_colormaps()
# ------------------

_signal_labels = signal_labels(absolute)   # shared with fig5 so the two cannot drift
lw = 1.8

# Heatmap data
_reg_deriv = results_reg[signal_type]['deriv']

if plot_level == 'region':
    sort_idx     = np.argsort(sa_axis_reg)
    deriv_sorted = _reg_deriv[sort_idx]
    ylabel       = 'S-A axis'
    bottom_label = 'Sensorimotor pole'
    top_label    = 'Association pole'
else:
    deriv_sorted = results_dec[signal_type]['deriv']
    ylabel       = 'S-A axis decile'
    bottom_label = 'Sensorimotor'
    top_label    = 'Association'

n_rows = deriv_sorted.shape[0]
vmax   = np.nanpercentile(np.abs(deriv_sorted), 98)

# Panel E: observed r curve plus whichever significance verdict SIG_METHOD asks for -- both are
# per signal, because a band or a p-curve borrowed from another signal silently mislabels the
# shading and the crossing-age annotations.
r_vals = compute_deriv_sa_corr(_reg_deriv, sa_axis_reg, method=SA_CORR_METHOD)
_ci_lo, _ci_hi = r_ci_lo[signal_type], r_ci_hi[signal_type]
_null_p        = sa_null_p[signal_type]
_have_ci, _have_p = _ci_lo is not None, _null_p is not None
if SIG_METHOD in ('bootstrap', 'both') and not _have_ci:
    print(f'NOTE: no bootstrap CI cached for {signal_type!r}; falling back to a bare curve.')
if SIG_METHOD in ('spatial', 'both') and not _have_p:
    print(f'NOTE: no spatial-null p cached for {signal_type!r} (regenerate with '
          'FORCE_RERUN = True); falling back to the bootstrap verdict.')

# SA-decile 2nd derivative (acceleration) of the fit: central differences of pred with
# edge_order=2 (matches build_signal). edge_order=2 avoids the compounded boundary
# artifact a plain double np.gradient produces at the age extremes.
_pred_dec   = results_dec[signal_type]['pred']
_deriv2_dec = np.gradient(np.gradient(_pred_dec, age_grid, axis=1, edge_order=2),
                          age_grid, axis=1, edge_order=2)

# Data-driven change-point boundaries: draw one dashed vertical line per detected boundary
# (n_bkps of them; a single line when n_bkps=1) on every panel. These are always detected on the
# deflection fit, so they read as a fixed reference regardless of signal_type.
def _draw_stage_boundary(ax, on_image=False):
    _z = None if on_image else 0        # behind the data curves on the line plots
    for _a in boundary_ages:
        ax.axvline(_a, color='0.2', linestyle='--', lw=0.9, alpha=0.9,
                   **({} if _z is None else {'zorder': _z + 0.5}))

# Bold panel label (A, B, ...) at each subplot's top-left corner.
def _panel_label(ax, letter):
    ax.annotate(letter, xy=(0.0, 1.0), xycoords='axes fraction', xytext=(-22, 6),
                textcoords='offset points', fontsize=11, fontweight='bold',
                va='bottom', ha='left', annotation_clip=False)

# Figure layout
fig      = plt.figure(figsize=(8.5, 6.5), dpi=300)
gs_outer = GridSpec(1, 2, figure=fig, width_ratios=[1, 1], wspace=0.15)

# Left column: heatmap | narrow vertical colorbar
gs_left   = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_outer[0],
                                    width_ratios=[1, 0.025], wspace=0.05)
ax_heat   = fig.add_subplot(gs_left[0])
ax_cbar_h = fig.add_subplot(gs_left[1])

# Right column: pred / deriv1 / deriv2 / Pearson r (4 panels; SA-decile colorbar added as overlay)
gs_right = GridSpecFromSubplotSpec(4, 1, subplot_spec=gs_outer[1],
                                   height_ratios=[1, 1, 1, 1], hspace=0.3)
ax0 = fig.add_subplot(gs_right[0])
ax1 = fig.add_subplot(gs_right[1], sharex=ax0)
ax2 = fig.add_subplot(gs_right[2], sharex=ax0)
ax3 = fig.add_subplot(gs_right[3], sharex=ax0)

# Heatmap
im = ax_heat.imshow(
    deriv_sorted, aspect='auto', origin='lower',
    extent=[age_grid[0], age_grid[-1], 0, n_rows],
    cmap=heatmap_cmap, vmin=-vmax, vmax=vmax, interpolation='nearest',
)
_draw_stage_boundary(ax_heat, on_image=True)

ax_heat.set_xlabel('Age (years)', fontsize=8, labelpad=2)
ax_heat.set_ylabel(ylabel, fontsize=8, labelpad=4)
if plot_level == 'region':
    ax_heat.set_yticks([0, n_rows // 2, n_rows])
    ax_heat.set_yticklabels(['', '', ''], fontsize=8)
else:
    ax_heat.set_yticks(np.arange(n_rows) + 0.5)
    ax_heat.set_yticklabels([str(i + 1) for i in range(n_rows)], fontsize=8)
ax_heat.tick_params(labelsize=8, pad=1)

ax_heat.annotate(top_label,    xy=(age_grid[0], n_rows), xytext=(-8, -1.5), rotation=90,
                 textcoords='offset points', fontsize=8, va='top',    annotation_clip=False)
ax_heat.annotate(bottom_label, xy=(age_grid[0], 0),      xytext=(-8,  1.5), rotation=90,
                 textcoords='offset points', fontsize=8, va='bottom', annotation_clip=False)
ax_heat.annotate('', xy=(age_grid[-1], n_rows + n_rows * 0.015),
                 xytext=(age_grid[0],  n_rows + n_rows * 0.015),
                 arrowprops=dict(arrowstyle='->', color='k', lw=0.8), annotation_clip=False)
ax_heat.text((age_grid[0] + age_grid[-1]) / 2, n_rows + n_rows * 0.025,
             'Age-resolved analysis', ha='center', va='bottom', fontsize=8, clip_on=False)

cb_h = fig.colorbar(im, cax=ax_cbar_h, orientation='vertical')
cb_h.set_label(f'{_signal_labels[signal_type]}\n(1st derivative)', fontsize=8)
cb_h.ax.yaxis.set_label_position('right')
cb_h.ax.tick_params(labelsize=6)

# Line plots: pred (ax0), 1st derivative (ax1), 2nd derivative (ax2). style_ax(demean=...) only
# controls the dashed y=0 reference line: meaningful for the demeaned prediction panel, and always
# meaningful for the derivative panels (zero rate / zero acceleration), so those keep it on.
for i in draw_order:
    ax0.plot(age_grid, results_dec[signal_type]['pred'][i], color=series_colors[i], lw=lw)
style_ax(ax0, 'Age (years)', f'{_signal_labels[signal_type]}\n(prediction)', None, demean=demean)
ax0.set_xlabel('')
ax0.set_box_aspect(0.8)
plt.setp(ax0.get_xticklabels(), visible=False)

for i in draw_order:
    ax1.plot(age_grid, results_dec[signal_type]['deriv'][i], color=series_colors[i], lw=lw)
style_ax(ax1, 'Age (years)', f'{_signal_labels[signal_type]}\n(1st derivative)', None, demean=True)
ax1.set_xlabel('')
ax1.set_box_aspect(0.8)
plt.setp(ax1.get_xticklabels(), visible=False)

for i in draw_order:
    ax2.plot(age_grid, _deriv2_dec[i], color=series_colors[i], lw=lw)
style_ax(ax2, 'Age (years)', f'{_signal_labels[signal_type]}\n(2nd derivative)', None, demean=True)
ax2.set_xlabel('')
ax2.set_box_aspect(0.8)
plt.setp(ax2.get_xticklabels(), visible=False)

# r(SA axis × regional GAM derivative). The band is the subject bootstrap; the significance
# marks are whichever test SIG_METHOD names. 'both' draws the bootstrap band but marks
# significance from the spatial null, which is the conservative reading of the two.
_band = (_ci_lo, _ci_hi) if (_have_ci and SIG_METHOD in ('bootstrap', 'both')) else None
_sig  = (_null_p < SIG_ALPHA) if (_have_p and SIG_METHOD in ('spatial', 'both')) else None
_sig_used = plot_r_age(ax3, age_grid, r_vals, ci=_band, sig=_sig,
                       style='band' if _band is not None else 'trace', lw=lw)
_src = ('spatial null' if _sig is not None else
        'bootstrap CI' if _band is not None else 'none')
print(f'Panel E [{signal_type}]: significance from {_src}; '
      f'{int(_sig_used.sum())}/{len(age_grid)} age points marked.')
ax3.set_xlabel('Age (years)', fontsize=8, labelpad=1)
ax3.set_ylabel(f'Pearson $r$\n(1st deriv. × S-A axis)', fontsize=8, labelpad=1)
ax3.tick_params(labelsize=8, pad=0.25)
ax3.set_box_aspect(0.8)
ax3.spines[['top', 'right']].set_visible(False)

# tau line(s) on the right-column panels (matching the heatmap)
for _ax in (ax0, ax1, ax2, ax3):
    _draw_stage_boundary(_ax)

# Bold panel labels: A = heatmap, B-E = right column (top -> bottom)
for _ax, _lab in [(ax_heat, 'A'), (ax0, 'B'), (ax1, 'C'), (ax2, 'D'), (ax3, 'E')]:
    _panel_label(_ax, _lab)

# SA-decile colorbar on the right-hand side of the top-right panel (ax0)
fig.canvas.draw()
add_sa_decile_colorbar(fig, ax0, cmap, n_deciles, orientation='vertical')

# dpi=1000: panel A is a 400-row imshow, which needs more than the 300 dpi a line panel wants.
_path = save_panel(fig, nas_scratch_dir,
                   f'fig4_combined_{task_scan}_{signal_type}{variant_tag}_{SIG_METHOD}',
                   dpi=1000)
print(f'-> {os.path.basename(_path)}.{{png,svg}}')
plt.show()

# Figure 4 — supplement: S-A decile GAM fits


In [ ]:
# ── Supplementary: panel B on its own ────────────────────────────────────────────────────────────
# The fitted age curve for each S-A decile, coloured sensorimotor -> association, at full size and
# with the same stage-boundary annotation the main figure uses. Nothing is recomputed: these are the
# same cached decile fits panel B draws, so the two can never disagree.
# Follows signal_type / demean / variant_tag from the figure cell above, so re-running that cell
# with a different signal and then this one keeps the pair in step.
SUPP_PRED_FIGSIZE = (2.6, 2.2)     # inches (the axes; the colorbar is placed outside it)

fig, ax = plt.subplots(figsize=SUPP_PRED_FIGSIZE, dpi=300)
for i in draw_order:
    ax.plot(age_grid, results_dec[signal_type]['pred'][i], color=series_colors[i], lw=lw)
_draw_stage_boundary(ax)
# demean=demean: style_ax's dashed y=0 line only means something when the fits were centred.
style_ax(ax, 'Age (years)', f'{_signal_labels[signal_type]}\n(prediction)', None, demean=demean)
ax.set_box_aspect(0.8)

# The same S-A decile colorbar the main figure hangs off panel B. add_sa_decile_colorbar aligns to
# the axes' final position, so the draw() first is required, not decorative.
fig.canvas.draw()
add_sa_decile_colorbar(fig, ax, cmap, n_deciles, orientation='vertical')

_path = save_panel(fig, nas_scratch_dir,
                   f'fig4_supp_decile_pred_{task_scan}_{signal_type}{variant_tag}')
print(f'S-A decile GAM fits [{signal_type}, {n_deciles} deciles] '
      f'-> {os.path.basename(_path)}.{{png,svg}}')
plt.show()


# Figure 4 — supplement: per-decile GAM fit check

In [ ]:
# ── Supplementary: per-decile GAM fit sanity check ──────────────────────────
# Every S-A decile, laid out on SUPP_GRID: the GAM prediction over the raw decile data it was fit to,
# for the SAME signal_type the figure above plots, so the two are always read together. The raw
# decile values are recomputed here (mirroring the fit exactly), so this works whether the GAM cache
# was loaded or freshly computed. The curve is the age smooth evaluated at mean nuisance, so the raw
# data is put on the same footing as the fit: demeaned when demean=True (both centred on zero),
# left as-is when demean=False (both on the native scale). Extra scatter vs the curve is the
# nuisance covariates (sex, meanFD, mean rest INT) + noise, not lack of fit.
import matplotlib.patheffects as pe

SUPP_GRID       = (2, 5)      # (rows, cols) of decile panels
SUPP_PANEL_SIZE = (1.4, 1.5)  # size of ONE panel (inches)

_nrow, _ncol = SUPP_GRID
if _nrow * _ncol < n_deciles:
    raise ValueError(f'SUPP_GRID {SUPP_GRID} holds {_nrow * _ncol} panels but there are '
                     f'{n_deciles} deciles to show.')

_rest_dec = data_dict_centiles[rest_scan]
try:
    _task_dec = data_dict_centiles[task_scan]
except (KeyError, TypeError):
    _task_dec = data_dict_centiles[rest_scan]
# Raw decile data per signal, matching what fit_all_panels was handed in the fit cell.
raw_dec  = {'rest': _rest_dec, 'task': _task_dec,
            'deflection': compute_deflections(_rest_dec, _task_dec, absolute=absolute)}[signal_type]
age_subj = df_behavior['age'].values
pred_dec = results_dec[signal_type]['pred']        # (n_deciles, n_grid); demeaned iff demean

fig, axes = plt.subplots(_nrow, _ncol, dpi=300, sharex=True, squeeze=False,
                         figsize=(SUPP_PANEL_SIZE[0] * _ncol, SUPP_PANEL_SIZE[1] * _nrow))
for j in range(_nrow * _ncol):
    ax = axes.ravel()[j]
    if j >= n_deciles:                    # spare cells when the grid is larger than n_deciles
        ax.set_visible(False)
        continue
    y_raw = raw_dec[:, j].astype(float)
    if demean:
        y_raw = y_raw - np.nanmean(y_raw)                    # demean to match the fitted curve
    ax.scatter(age_subj, y_raw, s=5, color='0.6', alpha=0.35, edgecolors='none', zorder=1)
    # Curve coloured by S-A decile (series_colors), with a dark outline via path_effects so the
    # near-white mid-deciles stay visible against the white background.
    ax.plot(age_grid, pred_dec[j], color=series_colors[j], lw=2.0, zorder=3,
            path_effects=[pe.Stroke(linewidth=3.0, foreground='0.25'), pe.Normal()])
    if demean:
        ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4, zorder=2)   # only meaningful when centred
    # Per-panel limits, not shared: this is a FIT check, so each panel should show whether the curve
    # tracks its own data. Robust to outliers, which would otherwise flatten every curve to a line.
    lo, hi = np.nanpercentile(y_raw, [1, 99])
    ax.set_ylim(lo - 0.15 * (hi - lo), hi + 0.15 * (hi - lo))
    ax.set_title(f'S-A decile {j + 1}', fontsize=8)
    ax.tick_params(labelsize=7, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
    if j // _ncol == _nrow - 1:           # x label on the bottom row only (sharex hides the rest)
        ax.set_xlabel('Age (years)', fontsize=8, labelpad=2)
    if j % _ncol == 0:                    # y label on the leftmost column of each row
        ax.set_ylabel(_signal_labels[signal_type], fontsize=8)
# fig.suptitle(f'Per-decile GAM fit ({signal_type}; {fit_recipe}, n_splines={n_splines}, '
#              f'absolute={absolute}, demean={demean})', fontsize=8, y=1.00)

fig.tight_layout()
_path = save_panel(fig, nas_scratch_dir,
                   f'fig4_supp_decile_fits_{task_scan}_{signal_type}{variant_tag}')
print(f'{n_deciles} deciles on a {_nrow}x{_ncol} grid -> {os.path.basename(_path)}.{{png,svg}}')
plt.show()
